In [1]:
import re, os
import numpy as np
import pandas as pd
from time import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np


In [2]:
import json, re, os, numpy as np, pandas as pd

PATH = "full_news_data.jsonl"

rows = []
with open(PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try:
            rows.append(json.loads(line))
        except Exception:
            pass

df = pd.DataFrame(rows)
for col in ["title","content","source","label","author","publication_date","url"]:
    if col not in df.columns: df[col] = None

# unify labels then filter to just Democrat/Republican
df["label"] = df["label"].replace({"Other":"Other/Neutral","Neutral":"Other/Neutral"})
df = df[df["label"].isin(["Democrat","Republican"])].copy()

# light de-boiler (helps DailyMail/Federalist)
def clean_boiler(text: str) -> str:
    if not text: return ""
    t = text
    t = re.sub(r"Published:\s*\d{1,2}:\d{2}\s*(AM|PM)?\s*.*?(View\s*comments|$)", " ", t, flags=re.I|re.S)
    t = re.sub(r"Share what you think.*?(Privacy Policy|Published by|$)", " ", t, flags=re.I|re.S)
    t = re.sub(r"(Follow .*? Twitter @\w+\.?)", " ", t, flags=re.I)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["_len"] = df["content"].fillna("").map(lambda s: len(s.split()))
df = df[df["_len"] >= 80].copy()

df["title"] = df["title"].fillna("").astype(str)
df["content"] = df["content"].fillna("").astype(str).map(clean_boiler)

# content-only text (no source tokens)
df["text"] = (
    df["title"].fillna("") + " " + df["content"].fillna("")
)


print("Label counts:\n", df["label"].value_counts())
print("Source counts (top):\n", df["source"].fillna("UNKNOWN").value_counts().head(10))
df.head(2)


Label counts:
 label
Republican    1000
Democrat       903
Name: count, dtype: int64
Source counts (top):
 source
CNN           903
FEDERALIST    500
DAILYMAIL     500
Name: count, dtype: int64


,title,source,author,publication_date,content,url,label,_len,text
0,Krugman vs. “The Crazy Party”,FEDERALIST,David Harsanyi,2013-09-20T00:00:00,Politics Krugman vs. “The Crazy Party” By: Dav...,https://thefederalist.com/2013/09/20/krugman-v...,Republican,776,Krugman vs. “The Crazy Party” Politics Krugman...
1,"Now, end the witch hunt! Campaigners' demands ...",DAILYMAIL,"MARK NICOL, DEFENCE EDITOR",2025-10-23T18:29:00,"By GLEN KEOGH IN BELFAST andMARK NICOL, DEFENC...",https://www.dailymail.co.uk/news/article-15222...,Republican,1143,"Now, end the witch hunt! Campaigners' demands ..."


In [3]:
import numpy as np, pandas as pd

# PARAMETERS — tweak as you like
HOLDOUT_FRAC = 0.20     # or set HOLDOUT_N instead of fraction
HOLDOUT_N    = None     # e.g., 150  (takes precedence if not None)
RANDOM_STATE = 42

# Sanity: only Democrat/Republican data should already be filtered in df
assert set(df["label"].unique()) <= {"Democrat","Republican"}, "Filter df to D/R first."

# make holdout from democrat articles using cnn articles
cnn_mask = (df["source"] == "CNN") & (df["label"] == "Democrat")
cnn_df = df[cnn_mask].copy()
if len(cnn_df) == 0:
    raise ValueError("No CNN Democrat rows available for holdout.")

if HOLDOUT_N is not None:
    n = min(HOLDOUT_N, len(cnn_df))
    holdout_df = cnn_df.sample(n=n, random_state=RANDOM_STATE)
else:
    holdout_df = cnn_df.sample(frac=HOLDOUT_FRAC, random_state=RANDOM_STATE)

train_df = df.drop(index=holdout_df.index).reset_index(drop=True)
holdout_df = holdout_df.reset_index(drop=True)

# also hold out for Republican data
rep_mask = (df["source"] == "FEDERALIST") & (df["label"] == "Republican")
rep_holdout = df[rep_mask].sample(frac=0.2, random_state=42)
train_df = df.drop(index=rep_holdout.index.union(holdout_df.index))
full_holdout = pd.concat([holdout_df, rep_holdout])

print(full_holdout["label"].value_counts())

label
Democrat      181
Republican    100
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

def balance_by_label_source(df_in, per_group_max=300, seed=42):
    g = (df_in.groupby(["label","source"], dropna=False, group_keys=False)
               .apply(lambda x: x.sample(min(len(x), per_group_max), random_state=seed)))
    return g.sample(frac=1.0, random_state=seed).reset_index(drop=True)

balanced_train_df = balance_by_label_source(train_df, per_group_max=300)
print("Balanced train size:", len(balanced_train_df))
print(balanced_train_df.groupby(["label","source"]).size().head(12))

labels = ["Democrat","Republican"]  # fixed order for binary


Balanced train size: 900
label       source    
Democrat    CNN           300
Republican  DAILYMAIL     300
            FEDERALIST    300
dtype: int64


/var/folders/lw/gtrydkx90_j86z8df4wls0r80000gn/T/ipykernel_29680/2606591136.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), per_group_max), random_state=seed)))


In [5]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

def make_pipeline():
    word_tfidf = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1,3),
        min_df=3, max_df=0.9, max_features=200_000,
    )
    char_tfidf = TfidfVectorizer(
        analyzer="char",
        ngram_range=(3,5),
        min_df=5, max_df=0.95,
        lowercase=False,
    )
    feats = FeatureUnion([("w", word_tfidf), ("c", char_tfidf)])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced",
                             solver="liblinear", multi_class="ovr")
    return Pipeline([("feats", feats), ("clf", clf)])

# (optional) internal validation split from training set for quick monitoring
labels = ["Democrat","Republican"]
sub_tr, sub_val = train_test_split(
    balanced_train_df, test_size=0.15, random_state=42, stratify=balanced_train_df["label"]
)

pipe = make_pipeline()
pipe.fit(sub_tr["text"].values, sub_tr["label"].values)

pred_val = pipe.predict(sub_val["text"].values)
print("== Validation on train split ==")
print(classification_report(sub_val["label"].values, pred_val, digits=3))
print("Confusion matrix:\n", confusion_matrix(sub_val["label"].values, pred_val, labels=labels))


/Users/selinawu/miniforge3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


== Validation on train split ==
              precision    recall  f1-score   support

    Democrat      0.978     1.000     0.989        45
  Republican      1.000     0.989     0.994        90

    accuracy                          0.993       135
   macro avg      0.989     0.994     0.992       135
weighted avg      0.993     0.993     0.993       135

Confusion matrix:
 [[45  0]
 [ 1 89]]


In [6]:
pred_hold = pipe.predict(full_holdout["text"].values)
print(classification_report(full_holdout["label"].values, pred_hold, digits=3))
print("Confusion matrix:\n", confusion_matrix(full_holdout["label"].values, pred_hold, labels=labels))


              precision    recall  f1-score   support

    Democrat      1.000     1.000     1.000       181
  Republican      1.000     1.000     1.000       100

    accuracy                          1.000       281
   macro avg      1.000     1.000     1.000       281
weighted avg      1.000     1.000     1.000       281

Confusion matrix:
 [[181   0]
 [  0 100]]


In [7]:
from joblib import dump, load
import json, numpy as np, re, os

os.makedirs("models", exist_ok=True)
dump(pipe, "models/party_affil_DR_union_cnnHoldout.joblib")
with open("models/party_labels.json", "w") as f:
    json.dump({"classes": labels}, f)
print("Saved: models/party_affil_DR_union_cnnHoldout.joblib")

# --- prediction helper (content-only) ---
def clean_boiler(text: str) -> str:
    if not text: return ""
    t = re.sub(r"Published:\s*\d{1,2}:\d{2}\s*(AM|PM)?\s*.*?(View\s*comments|$)", " ", text, flags=re.I|re.S)
    t = re.sub(r"Share what you think.*?(Privacy Policy|Published by|$)", " ", t, flags=re.I|re.S)
    t = re.sub(r"(Follow .*? Twitter @\w+\.?)", " ", t, flags=re.I)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def prepare_text(article: dict) -> str:
    title = (article.get("title") or "").strip()
    content = (article.get("content") or "").strip()
    return (title + "\n\n" + clean_boiler(content)).strip()

def predict_article_DR(article: dict, model_path="models/party_affil_DR_union_cnnHoldout.joblib"):
    mdl = load(model_path)
    with open("models/party_labels.json") as f:
        classes = json.load(f)["classes"]  # ["Democrat","Republican"]
    text = prepare_text(article)
    proba = mdl.predict_proba([text])[0]
    pred_idx = int(np.argmax(proba))
    return classes[pred_idx], {cls: float(p) for cls, p in zip(classes, proba)}

# Example
example = {
    "title": "Trump 'retired' a database tracking the most expensive weather disasters...",
    "source": "CNN",
    "author": "Andrew Freedman",
    "publication_date": "2025-10-22 10:00:00",
    "content": "The Billion-Dollar Weather and Climate Disasters Database, which the Trump administration..."
}
pred, probs = predict_article_DR(example)
print("Prediction:", pred)
print("Probabilities:", probs)


Saved: models/party_affil_DR_union_cnnHoldout.joblib
Prediction: Democrat
Probabilities: {'Democrat': 0.5834732304420793, 'Republican': 0.4165267695579206}


In [8]:
article = {
    "title":  "How Kyrsten Sinema’s decision makes Democrats’ 2024 Senate map tighter",
    "source": "CNN",
    "author": "Harry Enten",
    "publication_date": "2025-10-22 10:00:00",
    "content": "Arizona Sen. Kyrsten Sinema decided to shake up the political world on Friday by becoming an independent. The former Democrat is still caucusing with the party in the Senate, so the Democratic caucus still has 51 members. Now, instead of 49 Democrats and two independents within their ranks, the caucus has 48 Democrats and three independents.\nBut that simple math hides a more clouded picture for Democrats and for Sinema herself. Sinema’s interests are no longer necessarily the Democrats’ best interests in the next Congress, and the 2024 Senate map became even more complicated for Democrats with Sinema’s decision.\nTo be clear, Sinema has always been a thorn in the Democrats side during her time in Congress. Over the last two years, Democrats have had to almost always make sure that any bill or nomination had Sinema’s support to have any chance of passing. That’s the math when you have only 50 Senate seats in a 100-seat chamber. A lot of bills and nominations were never voted on without Sinema and Manchin’s backing.\nFrom 2013 (Sinema’s first term in Congress) to 2020, Sinema voted against her party more than almost any other member of Congress. She stayed with the party about 69% of the time on votes where at least one half of the Democrats voted differently than half of Republicans. The average Democrat voted with their party about 90% of the time on these votes.\nIt’s quite possible that Sinema’s percentage of sticking with the party will lower now that she is an independent. Consider the example of former Sen. Joe Lieberman. The longtime Democrat won reelection as a third-party candidate in 2006, after losing the Democratic primary to a left-wing challenger (the now fairly moderate Connecticut Gov. Ned Lamont)\nRelative to the average Senate Democrat, Lieberman voted with the party 10 points less of the time after becoming an independent than he had in his last term as a Democrat. If that happens with Sinema, she’ll become even more conservative than West Virginia’s Joe Manchin (the most conservative member of the Democratic caucus).\nThis would make sense because the incentive structure is now very different for Sinema. Ahead of a 2024 reelection campaign, she no longer has to worry about winning a Democratic primary. Sinema has to worry about building a coalition of Democrats, independents and Republicans. That is far more difficult to do if you’re seen as too liberal.\nIndeed, the big reason Sinema became an independent is because it would have been very difficult to win a Democratic primary. Her approval rating among Arizona Democrats in an autumn 2022 CES poll stood at just 25%. A number of Democrats (e.g. Rep. Ruben Gallego and Rep. Greg Stanton) were already lining up to potentially challenge her in a primary.\nA question now is whether Sinema’s decision to become an independent will dissuade some of those Democrats from running. The idea being that Sinema still caucuses with the Democrats, and Democrats wouldn’t want to split the Democratic vote in a general election allowing a Republican to win in a purple state like Arizona.\nIt’s an interesting bet from Sinema. After all, Democrats usually don’t run a candidate against independent Sen. Bernie Sanders in Vermont. The Democrats who run against independent Sen. Angus King in Maine have not gained traction in recent elections. Don’t forget the aforementioned Lieberman won as a third-party candidate.\nThe electoral math structure was and is totally different in these circumstances, however. Sanders wouldn’t attract a left-wing Democratic challenger because he is already so progressive. Lieberman declared his third-party candidacy after the primary, so Republicans didn’t have time to find a well-known challenger. Republicans also knew that Lieberman, who was an ardent supporter of the Iraq War, was probably the best they could hope for in the deeply Democratic state of Connecticut.\nThis leaves the King example. King, like Sinema, is a moderate from not a deeply blue or red state. There’s just one problem for Sinema in this analogy: King is popular. He had previously won the governorship twice as an independent and has almost always sported high favorables.\nSinema is not popular at all. The CES poll had her approval rating below her disapproval rating with Democrats, independents and Republicans in Arizona. Sinema’s overall approval stood at 25% to a disapproval rating of 58%. Other polling isn’t nearly as dire for Sinema, but the average of it all has her firmly being more unpopular than popular.\nPut another way, Sinema’s current numbers are probably not going to scare off many challengers from either the Democratic or Republican side. Additionally, there’s zero reason for Democrats to cede the ground to Sinema because it would keep a Republican from winning. It isn’t clear at all that Sinema can win as an independent.\nWhat Sinema’s move did accomplish is that it made the electoral math a lot more complicated in Arizona and therefore nationally. Having two people in the race who are going to caucus with the Democratic Party likely makes it more difficult for the Democrats to win.\nOne potential worrisome example for Democrats in a purple state (at least then) was the 2010 Florida Senate race. Then Republican Gov. Charlie Crist decided to run as an independent after it became clear he wouldn’t beat the more conservative Republican Marco Rubio in a Republican primary. Crist, who said he would caucus with the Democrats, split the Democratic vote with then Rep. Kendrick Meek, and Rubio cruised to a win.\nI should point out that Democrats certainly have a chance. The 1968 Alaska Senate race, for example, featured two Democrats (Mike Gravel and then Sen. Ernest Gruening as write-in). Gravel won in the state which Republican Richard Nixon carried, too, by a few points.\nIn 2024, Arizona Republicans could nominate an extreme candidate that flames out. They just lost every major statewide race in 2022 because of who they nominated.\nDon’t dismiss the possibility too that Sinema could win like Harry Byrd did in the 1970 Virginia Senate election when both parties nominated candidates. Maybe voters will like Sinema’s new independent registration.\nSinema also could find herself flaming out when running in the general election without a major party backing her like Gruening did in 1968 or then Sen. Jacob Javits in the 1980 New York Senate race.\nWe just don’t know.\nAll that said, the Democrats already have a difficult map heading into 2024. Depending on whether the Democrats win the presidency (and have a Democratic vice president who can break Senate ties), they can afford to lose zero to one Senate seats and maintain a majority.\nThe vast majority, 23 of the 34, senators up for reelection in 2024 caucus with the Democrats. An abnormally large number (7) represent states Republican Donald Trump won at least once. This includes Arizona.\nWith Sinema’s break from the Democratic party, the road is, if nothing else, curvier for Democrats.", "url": "https://www.cnn.com/2022/12/10/politics/kyrsten-sinema-independent-democrats/index.html", "label": "Democrat"}

pred, probs = predict_article_DR(article)
print("Prediction:", pred)
print("Probabilities:", probs)


Prediction: Republican
Probabilities: {'Democrat': 0.43091509645588, 'Republican': 0.56908490354412}


In [9]:
article = {
    "title":  "Sparks fly as Cuomo, Mamdani tear into each other during fiery debate: 'Toxic energy'",
    "source": "FOX",
    "author": "Alec Schemmel",
    "publication_date": "2025-10-22T00:00:00",
    "content": "Front-runners for New York City mayor, Zohran Mamdani and Andrew Cuomo, wasted little time attacking each other on alleged personal scandals they have been involved in during a Wednesday night debate between the pair and GOP candidate Curtis Sliwa.\nMamdani and Sliwa took the opportunity during Wednesday's debate to drill down on past sexual harassment allegations against Cuomo, the former governor of New York, ahead of an impeachment inquiry that preceded Cuomo's 2021 resignation. Cuomo was also hit by Mamdani over accusations he has – while in public office – failed to meet with Muslim constituents and only began doing so amid pressure from his mayoral campaign, and over his alleged poor handling of the COVID-19 virus in New York after Cuomo was party to issuing guidance forcing nursing homes and long-term care facilities to admit COVID-19 positive patients.\nMeanwhile, Cuomo did not hold back on targeting Mamdani over alleged controversies that have embattled his campaign. Cuomo blasted the self-proclaimed socialist over his lack of experience, ties to radical politics, and past radical comments about law enforcement, Israel and the situation in Gaza.\nFBI AGENTS FROM '93 WTC ATTACK BLAST MAMDANI FOR EMBRACING RADICAL IMAM\n\"My main opponent has no new ideas. He has no new plan. … He's never run anything, managed anything. He's never had a real job,\" Cuomo said of Mamdani during the debate. Cuomo also branded Mamdani as someone who has proven to be \"a divisive force in New York,\" pointing to past incidents that have garnered Mamdani heat from critics.\nOne of those incidents included a picture he took with a hard-lined Ugandan lawmaker who has pushed policies of imprisoning people for being gay, which Mamdani took while taking a break from the campaign trail to visit his home country of Uganda for a wedding. Cuomo also hit the controversy over whether Mamdani supports Jewish New Yorkers, as his critics have claimed he is anti-Israel pointing to statements he has made, like \"globalize the intifada.\"\nCuomo also accused Mamdani of disrespecting Italian-Americans after a video of him surfaced giving the middle finger to a statue of Christopher Columbus, while also pointing to criticism the self-proclaimed socialist candidate has garnered from 9/11 first-responders after posting a photo with a Muslim cleric who served as a character witness for the mastermind behind the September 11, 2001 attacks.\nTOP 5 MOMENTS FROM FIERY NYC MAYORAL DEBATE: 'HE LITERALLY HAS NEVER HAD A JOB'\n\"You have been a divisive force in New York, and I believe that's toxic energy for New York. It's with the Jewish community. It's with the Italian-American community – when you give the Columbus statue the finger. It's with the Sunni Muslims when you say decriminalize prostitution, which is Haram. It's the Hindus,\" Cuomo continued. \"Then, you take a picture with Rebecca Kadaga, deputy Prime Minister of Uganda. … She's known as Rebecca ‘Gay Killer.’ … You're a citizen of Uganda. You took the picture. You said you didn't know who she was. It turns out you did. How do you not renounce your citizenship or demand BDS against Uganda for imprisoning people who are gay just by their sexual orientation? Isn't that a basic violation of human rights?\"\nMamdani shot back that his politics have remained \"consistent\" and that they are built on a belief in human rights for all people, including LGBTQ+ folks. Had he known Kadga's role in drafting legislation to imprison gay folks, Mamdani said, he never would have taken the picture.\n\"This constant attempt to smear and slander me is an attempt to also distract from the fact that, unlike myself, you do not actually have a platform or a set of policies,\" Mamdani shot back at Cuomo before introducing his own claims about the former governor regarding past accusations of sexual harassment.\nMAMDANI RIPPED BY RIVALS FOR UNPOPULAR STANCE DURING FIERY NYC DEBATE: 'YOU WON'T SUPPORT ISRAEL'\n\"Mr. Cuomo. In 2021, 13 different women who worked in your administration credibly accused you of sexual harassment. Since then, you have spent more than $20 million in taxpayer funds to defend yourself, all while describing these allegations as entirely political,\" Mamdani said while attacking Cuomo Wednesday night.\n\"You have even gone so far as to legally go after these women. One of those women, Charlotte Bennett, is here in the audience this evening. You sought to access her private gynecological records. She cannot speak up for herself because you lodged a defamation case against her. I, however, can speak. What do you say to the 13 women that you sexually harassed?\"\nCuomo, in 2021, was accused of multiple incidents of sexual harassment that preceded his resignation as governor that year. A subsequent report from New York Attorney General Letitia James confirmed Cuomo \"sexually harassed multiple women from 2013 through 2020,\" while in January 2024, the U.S. Department of Justice announced it had reached a nearly $500,000 settlement with Cuomo's executive office over one of the claims. However, no criminal charges were ever filed against Cuomo, with some district attorneys citing insufficient evidence.\nCLICK HERE TO GET THE FOX NEWS APP\nCuomo defended himself against Mamdani's accusations, noting the cases were eventually dropped, before returning to questions about Mamdani's alleged past.\nMeanwhile, Sliwa didn't skip an opportunity to slam Cuomo over the sexual assault allegations either, saying early in the debate during a discussion about homelessness that Cuomo \"fled\" the governor's office amid an impeachment inquiry that was investigating him.\n\"Andrew, you didn't ‘leave.’ You fled from being impeached by the Democrats in the state legislature,\" Sliwa began before getting into the homelessness issue, earning him a round-of-applause from the audience.\n\"'Leave?' You fled!\" Sliwa continued to applause. \"But let's get back on topic.\"", "url": "https://www.foxnews.com/politics/sparks-fly-cuomo-mamdani-tear-into-each-other-during-fiery-debate", "label": "Republican"}

pred, probs = predict_article_DR(article)
print("Prediction:", pred)
print("Probabilities:", probs)

Prediction: Republican
Probabilities: {'Democrat': 0.49706507105365005, 'Republican': 0.50293492894635}


In [10]:
article = {
    "title":  "The latest on Donald Trump’s many legal clouds",
    "source": "CNN",
    "author": "Dan Berman, Zachary B. Wolf",
    "publication_date": "2023-10-02T00:00:00",
    "content": "Former President Donald Trump has been campaigning in between his many different court appearances for much of the year.\nBut his decision to attend the first day of his $250 million civil fraud trial in New York created another opportunity to appear on camera from inside a courtroom when the judge allowed photographers to document the moment before proceedings got underway.\nKeeping track of the dizzying array of civil and criminal cases is a full-time job.\nHe is charged with crimes related to conduct:\n- Before his presidency – a hush money scheme that may have helped him win the White House in 2016.\n- During his presidency – his effort to stay in the White House by overturning the 2020 election.\n- After his presidency – his treatment of classified material and alleged attempts to hide it from the National Archives.\nTrump denies any wrongdoing and has pleaded not guilty in all of the criminal cases. He alleges a “witch hunt” against him. But each trial has its own distinct storyline to follow.\nHere’s an updated list of developments in Trump’s very complicated set of court cases, beginning with the one playing out in Manhattan this week.\nNew York state civil court: $250 million fraud case\nThe civil fraud trial, unlike Trump’s multiple criminal indictments, does not carry the danger of a felony conviction and jail time, but it could very well cost him some of his most prized possessions, including Trump Tower.\nNew York Attorney General Letitia James brought the $250 million lawsuit in September 2022, alleging that Trump and his co-defendants committed repeated fraud in inflating assets on financial statements to get better terms on commercial real estate loans and insurance policies.\nJudge Arthur Engoron has already ruled that Trump and his adult sons are liable for fraud for inflating the value of his golf courses, hotels and homes on financial statements to secure loans.\nThe trial portion of the case, playing out in court in Manhattan, will assess what damages will be levied against Trump and how Engoron’s decision to strip Trump of his New York business licenses will play out.\nFederal civil court in New York: E. Jean Carroll defamation lawsuit\nIn May, a federal jury in Manhattan found Trump sexually abused former advice columnist E. Jean Carroll in a luxury department store dressing room in the mid-1990s and awarded her about $5 million.\nA separate civil defamation lawsuit will only need to decide how much money Trump has to pay her. That case for January 15 – the same day Iowa Republicans will hold their caucuses, the first date on the presidential primary calendar.\nFederal criminal court in DC: 2020 election Interference\nIn August, Trump was indicted by a federal grand jury in special counsel Jack Smith’s investigation into the aftermath of the 2020 election. The former president was arraigned in a Washington, DC, courtroom, where he pleaded not guilty.\nThe case is based in part on a scheme to create slates of fake electors in key states won by President Joe Biden.\nIn late September, Judge Tanya Chutkan rejected Trump’s request that she recuse herself from the case. Chutkan, a Barack Obama appointee, has overseen civil and criminal cases related to the January 6, 2021, insurrection and has repeatedly exceeded what prosecutors have requested for convicted rioters’ prison sentences.\nChutkan set the trial’s start date for March 4, 2024, the day before Super Tuesday, when the largest batch of presidential primaries will occur. The trial marks the first of Trump’s criminal cases expected to proceed.\nNew York criminal court: Hush money payments\nTrump has been charged in Manhattan criminal court with 34 felony counts of falsifying business records related to his role in a hush money payment scheme involving adult film actress Stormy Daniels late in the 2016 presidential campaign.\nThe former president pleaded not guilty at his April arraignment in Manhattan.\nProsecutors, led by Manhattan District Attorney Alvin Bragg, accuse Trump of falsifying business records with the intent to conceal $130,000 in payments to Daniels made by former Trump attorney and fixer Michael Cohen to guarantee her silence about an alleged affair.\nTrump has denied having an affair with Daniels.\nThe trial was originally scheduled to begin in late March 2024, but Judge Juan Merchan has suggested the date could move. The next court date is scheduled for February.\nGeorgia criminal court: Efforts to overturn election results\nFulton County District Attorney Fani Willis is using racketeering violations to charge a broad criminal conspiracy against Trump and 18 others in their efforts to overturn Biden’s victory in Georgia.\nThe probe was launched in 2021 following Trump’s call that January with Georgia Secretary of State Brad Raffensperger, in which the president pushed the Republican official to “find” votes to overturn the election results.\nThe August indictment also includes how Trump’s team allegedly misled state officials in Georgia; organized fake electors; harassed an election worker; and breached election equipment in rural Coffee County, Georgia.\nOne co-defendant, bail bondsman Scott Hall, has pleaded guilty to five counts in the case.\nFulton County prosecutors have signaled they could offer plea deals to other co-defendants.\nWillis this week issued a subpoena to former New York City Police Commissioner Bernard Kerik, a Trump ally, who in turn demanded an immunity deal in exchange for testimony.\nTrial for two co-defendants is expected to begin this month and could last three to five months. A trial date has not been set for Trump, who has pleaded not guilty.\nFederal criminal court in Florida: Mishandling classified material\nTrump has pleaded not guilty to 37 federal charges brought by Smith over his alleged mishandling of classified documents. Smith added three additional counts in a superseding indictment.\nThe investigation centers on sensitive documents that Trump brought to his Mar-a-Lago residence in Florida after his White House term ended in January 2021.\nThe National Archives, charged with collecting and sorting presidential material, has previously said that at least 15 boxes of White House records were recovered from Mar-a-Lago, including some classified records.\nTrump was also caught on tape in a 2021 meeting in Bedminster, New Jersey, where the former president discussed holding secret documents he did not declassify.\nSmith’s additional charges allege that Trump and his employees attempted to delete Mar-a-Lago security footage sought by the grand jury investigating the mishandling of the records.\nTrial is not expected until May, after most presidential primaries have concluded.\nThere are other cases to note:\nTrump Organization: Convicted of criminal tax fraud\nTrump’s namesake business, the Trump Organization, was convicted in December by a New York jury of tax fraud, grand larceny and falsifying business records in what prosecutors say was a 15-year scheme to defraud tax authorities by failing to report and pay taxes on compensation provided to employees.\nManhattan prosecutors told a jury the case was about “greed and cheating,” laying out a scheme within the Trump Organization to pay high-level executives in perks such as luxury cars and apartments without paying taxes on them.\nFormer Trump Organization Chief Financial Officer Allen Weisselberg pleaded guilty to his role in the tax scheme. He was released after serving four months in jail at Rikers Island.\nJanuary 6: Lawsuits by police officers\nSeveral members of the US Capitol Police and Washington, DC, Metropolitan Police are suing Trump, saying his words and actions incited the 2021 riot.\nThe various cases accuse Trump of directing assault and battery; aiding and abetting assault and battery; and violating Washington laws that prohibit the incitement of riots and disorderly conduct.\nIn August, Trump requested to put on hold the lawsuit related to the death of Capitol Police Officer Brian Sicknick, citing his various criminal trials. The estate of Sicknick, who died after responding to the attack on the Capitol, is suing two rioters involved in the attack and Trump for his alleged role in egging it on.\nOther lawsuits have been put on hold while a federal appeals court considers whether Trump had absolute immunity as the sitting president.\nPersonal retaliation: Peter Strzok lawsuit\nFormer top FBI counterintelligence official Peter Strzok, who was fired in 2018 after the revelation that he criticized Trump in text messages, sued the Justice Department, alleging he was terminated improperly.\nIn summer 2017, former special counsel Robert Mueller removed Strzok from his team investigating Russian interference in the 2016 election after an internal investigation revealed texts with former FBI lawyer Lisa Page that could be read as exhibiting political bias.\nStrzok and Page were constant targets of verbal attacks by Trump and his allies, part of the larger ire the then-president expressed toward the FBI during the Russia investigation. Trump repeatedly and publicly called for Strzok’s ouster until he was fired in August 2018.\nTrump is set to be deposed this month as part of the case, according to Politico.\nTrump lawsuit against Hillary Clinton, Democrats, ex-FBI officials is dismissed\nA federal judge dismissed Trump’s lawsuit against Hillary Clinton, the Democratic National Committee, several ex-FBI officials and more than two dozen other people and entities that he claims conspired to undermine his 2016 campaign with fabricated information tying him to Russia.\n“What (Trump’s lawsuit) lacks in substance and legal support it seeks to substitute with length, hyperbole, and the settling of scores and grievances,” US District Judge Donald Middlebrooks wrote.\nTrump appealed the decision, but Middlebrooks also ruled that the former president and his attorneys are liable for nearly $1 million in sanctions for bringing the case.\nTrump launched a Hail Mary bid in July to revive the sprawling lawsuit, relying on a recent report from special counsel John Durham that criticized the FBI’s Trump-Russia probe.\nTrump victory in Michael Cohen’s retaliation suit\nTrump’s former lawyer Cohen sued Trump, former Attorney General William Barr and others, alleging they put him back in jail to prevent him from promoting his upcoming book while under home confinement.\nCohen was serving the remainder of his sentence for lying to Congress and campaign violations at home, due to Covid-19 concerns, when he started an anti-Trump social media campaign in summer 2020. Cohen said that he was sent back to prison in retaliation and that he spent 16 days in solitary confinement.\nA federal judge threw out the lawsuit in November. District Judge Lewis Liman said he was empathetic to Cohen’s position but that Supreme Court precedent bars him from allowing the case to move forward.\nTrump-filed lawsuits: Bob Woodward\nTrump sued journalist Bob Woodward in January for alleged copyright violations, claiming Woodward released audio from their interviews without Trump’s consent.\nWoodward and publisher Simon & Schuster said Trump’s case is without merit and moved for its dismissal.\nWoodward conducted several interviews with Trump for his book “Rage,” published in September 2020. Woodward later released “The Trump Tapes,” an audiobook featuring eight hours of raw interviews with Trump interspersed with the author’s commentary.\nTrump-filed lawsuits: The New York Times, Mary Trump and CNN\nA federal judge in Florida dismissed Trump’s $475 million lawsuit against CNN that accused the network of defaming him by using the phrase “Big Lie” and allegedly comparing him to Adolf Hitler.\nA New York judge dismissed The New York Times from Trump’s lawsuit regarding disclosure of his tax returns and ordered Trump to pay the newspaper’s legal fees. Trump is still suing his niece Mary Trump for disclosure of the tax documents. She had tried to sue him for defrauding her out of millions after the death of his father, but the suit was dismissed.\nCNN’s Kara Scannell, Tierney Sneed, Katelyn Polantz and Marshall Cohen contributed to this report.", "url": "https://www.cnn.com/2022/11/15/politics/donald-trump-investigations-lawsuits/index.html", "label": "Democrat"}

pred, probs = predict_article_DR(article)
print("Prediction:", pred)
print("Probabilities:", probs)

Prediction: Democrat
Probabilities: {'Democrat': 0.89759209513965, 'Republican': 0.10240790486034998}


In [11]:
article = {
    "title":  "Merkley nearly breaks Booker's filibuster record, wins his praise for fighting 'Trump's authoritarian tactics'",
    "source": "FOX",
    "author": "Leo Briceno",
    "publication_date": "2025-10-22T00:00:00",
    "content": "Democrats pulled out all the stops Wednesday to delay the vote on a short-term spending bill to reopen the government, the 12th time the Senate has considered the measure since the government entered a shutdown Oct. 1.\nSen. Jeff Merkley, D-Ore., embarked on a nearly 24-hour speech at 6:23 p.m. Tuesday, concluding his remarks at 5 p.m. the next day. Merkley, 68, warned viewers of the authoritarianism he said had become a facet of the Trump administration.\n\"Be aware and worried about the possibility of the use of an emergency in order to expand authoritarian power. That’s the position we’re in now in the United States of America. Authoritarianism with a rubber-stamp Congress, a court that’s delivering more and more power to the executive and an executive who has a well-planned strategy,\" Merkley said in his remarks.\nJOHNSON WARNS US 'BARRELING TOWARD ONE OF THE LONGEST SHUTDOWNS' IN HISTORY\n\"Republicans have shut down the government to continue the strategy of slashing Americans' healthcare.\"\nHe delivered his speech as lawmakers remain gridlocked over federal funding for 2026. While Republicans in the House of Representatives have passed a short-term funding bill to keep the government open through Nov. 21, Democrats in the Senate have voted a dozen times to defeat the package.\nThe Senate once again failed to advance the package on Wednesday. It failed in a 54-46 vote.\nDemocrats, led by Senate Majority Leader Chuck Schumer, D-N.Y., and House Minority Leader Hakeem Jeffries, D-N.Y., have demanded an extension of COVID-era supplemental funding for Obamacare healthcare subsidies that will sunset in 2025.\nSCREAMING MATCH ERUPTS BETWEEN HAKEEM JEFFRIES, MIKE LAWLER AS GOVERNMENT SHUTDOWN CHAOS CONTINUES\nRepublicans need the support of seven Democrats to overcome the 60-vote threshold to overcome a filibuster. The GOP holds 53 seats in the chamber.\nMerkley, who came close to breaking Sen. Cory Booker’s 25-hour and 4-minute record set earlier this year, put the shutdown blame on Republicans throughout his discourse.\nBooker praised Merkley's stalling efforts online.\n\"Listening to Senator Jeff Merkley for over 22 hours, it is clear that we need to stand up for our democracy. We must continue to call out and counter Trump's authoritarian tactics. Thank you, Jeff!\" Booker said in a post on X.\nOn the issue of authoritarianism, the primary topic of Merkley's remarks, Merkley decried what he saw as the Trump administration’s attempts to push the limits on executive power, like its deployment of the National Guard to urban areas.\n\"If you remove a clear standard as to whether there is a rebellion and just say a president can deploy the military on a whim in places he doesn’t like against peaceful protesters to distract Americans or to exercise a suppression of dissent, then you have flung the doors open to tyranny, to a strongman state,\" Merkley said.\nCLICK HERE TO GET THE FOX NEWS APP\nPresident Donald Trump has deployed the National Guard to Washington, D.C.; Los Angeles; Chicago; Memphis; and Portland, Oregon, citing a need to protect law enforcement and government operations in those cities.", "url": "https://www.foxnews.com/politics/merkley-nearly-breaks-bookers-filibuster-record-wins-his-praise-fighting-trumps-authoritarian-tactics", "label": "Republican"}


pred, probs = predict_article_DR(article)
print("Prediction:", pred)
print("Probabilities:", probs)

Prediction: Democrat
Probabilities: {'Democrat': 0.6755520627782141, 'Republican': 0.32444793722178594}


In [12]:
article = {
    "title": "Texas Rep Jasmine Crockett weighs Senate bid after redistricting threatens House seat", 
    "source": "FOX", 
    "author": "Greg Wehner", 
    "publication_date": "2025-10-22T00:00:00", 
    "content": "Rep. Jasmine Crockett, D-Texas, said Wednesday she would \"strongly consider\" a run for the U.S. Senate, telling SiriusXM’s \"The Lurie Daniel Favors Show\" that recent polling suggests she would be a top contender in the Texas Democratic primary.\nDuring the interview, Favors asked Crockett about her district redrawn by the Texas legislature and whether she planned to continue running to represent her new district.\nCrockett said she had no idea where she might run, noting that she and her team are waiting for the court to issue a final ruling on whether the new district lines will go into effect. If the lines change, she said, she will be moved to another district.\n\"The other option is every other day there's a poll that comes out that makes it clear that I can win the primary for the U.S. Senate race in Texas, and I am looking, because if you want to take my seat of 766,000 away, I feel like there has to be some karma in that to where I take your seat that is for 30 million away,\" Crockett said.\nTEXAS GOP'S NEW MAP WOULD MOVE REP. CROCKETT’S HOME OUT OF HER DISTRICT, SLASH DEM SEATS\n\"The primary is the primary. That's cool. But, you gotta win in general, so we are doing some testing here shortly to see if I can expand the electorate,\" she added.\nCrockett said she does not put much stock in traditional polling, arguing it fails to capture voters who don’t typically participate in elections. She added that both Barack Obama and President Donald Trump succeeded by inspiring nontraditional voters, saying pollsters often overlook those who might be motivated to vote for the first time.\nShe said the key to winning in Texas is not relying on the existing electorate but expanding it. Crockett told the host she is studying polling data to identify which demographics could be mobilized and said that if she believes the electorate can be broadened enough, she would \"strongly consider\" entering the U.S. Senate race.\nFox News Digital reached out to Crockett’s team for comment on the matter.\nGov. Greg Abbott, R-Texas, signed a new congressional map into law in August, securing an additional five Republican-leaning U.S. House districts ahead of competitive midterm elections expected in 2026.\nThe Republican-controlled Texas House and Senate passed the new map through their respective chambers last week, following weeks of Lone Star State Democrats breaking quorum and fleeing the state to avoid a redistricting vote.\nCROCKETT CRITICIZES AOC AND BERNIE SANDERS' 'FIGHTING OLIGARCHY' TOUR FOR MAKING IT ABOUT THEMSELVES\nCrockett’s comments come after a year of statements that have drawn controversy.\nLast month, she was a guest on CNN’s \"The Arena\" when she said critics who believe \"Hitler\" and \"fascist\" comparisons contribute to political violence were \"absolutely wrong.\"\nHost Kasie Hunt asked Crockett during the show to respond to critics who pointed fingers at Democrats who use those terms and asked whether she thought they contribute to political violence.\n\"They’re absolutely wrong,\" Crockett said. \"Here’s the reality: They don’t want the American people to know any forms of history. We know that there was news out today about the president going after yet more historical information.\n\"The reality is that when we look at what is taking place, when you look at an authoritarian, and what they do is they try to basically say you have to do whatever the government says, even if that means that your personal freedoms are going to be subjected to whatever we say, whether it’s right or wrong. Right now, our personal freedoms are constantly under attack,\" she said.\nCrockett also appeared to mock Abbott, who is paralyzed from the waist down and uses a wheelchair, while speaking at a Human Rights Campaign event in March.\n\"Y’all know we got Governor Hot Wheels down there – come on now! And the only thing hot about him is that he is a hot a-- mess, honey,\" Crockett said.\nCLICK HERE TO DOWNLOAD THE FOX NEWS APP\nThe remark resurfaced as Crockett was already facing heavy criticism for other recent statements calling for Elon Musk to be \"taken down\" and for Sen. Ted Cruz, R-Texas, to be \"knocked over the head, like hard.\"\nFox News Digital’s Peter Pinedo, Deirdre Heavey and Alex Miller contributed to this report.", "url": "https://www.foxnews.com/politics/texas-rep-jasmine-crockett-weighs-senate-bid-after-redistricting-threatens-house-seat"}

pred, probs = predict_article_DR(article)
print("Prediction:", pred)
print("Probabilities:", probs)

Prediction: Republican
Probabilities: {'Democrat': 0.43591217617918543, 'Republican': 0.5640878238208146}


In [13]:
article = {
    "title": "Maine Dem Senate hopeful backed by Bernie Sanders apologizes for Nazi-style tattoo, vows to stay in race", 
    "source": "FOX", 
    "author": "Jasmine Baehr", 
    "publication_date": "2025-10-22T00:00:00", 
    "content": "Maine Democrat Graham Platner, a first-time Senate candidate backed by Sen. Bernie Sanders, I-Vt., says he has covered up a tattoo widely recognized as a Nazi symbol after critics unearthed old social media posts and demanded he quit the race.\nPlatner’s campaign is facing intense scrutiny after it was revealed he once had a skull-and-crossbones tattoo resembling the Totenkopf used by Hitler’s SS paramilitary forces.\nPlatner said he got the tattoo in 2007 during a \"night of drinking\" while on leave in Croatia in the Marine Corps and claimed he did not know its historical associations at the time. He has since covered the image with another tattoo.\nDELETED POSTS URGING VIOLENCE HAUNT DEMOCRATIC SENATE HOPEFUL IN MAINE RACE\nIn a video posted to Instagram Wednesday afternoon, Platner elaborated that the design was chosen from a flash tattoo wall while \"carousing\" with fellow Marines in Split, Croatia.\n\"We thought it looked cool,\" he said.\nHe claimed he had \"lived a life dedicated to anti-fascism, anti-racism and anti-Nazism\" and was \"appalled\" to learn it resembled a hate symbol.\nPlatner said he had never been questioned about the tattoo during his service and passed Army background checks.\nHe told The Associated Press he chose to cover rather than remove the tattoo due to a lack of removal services near his home in rural Maine.\n\"Going to a tattoo removal place is going to take a while,\" Platner said. \"I wanted this thing off my body.\"\nIn the video, Platner said he had the symbol inked over with a Celtic knot and imagery of dogs, a tribute to his family pets.\n\"This far more represents who I am now than even the skull and crossbones did,\" he said, lifting his shirt to reveal the new tattoo.\nREPUBLICAN LAWMAKER DIRECTS INVESTIGATION AFTER SWASTIKA VANDALISM DISCOVERED IN DC OFFICE\nThe controversy comes on the heels of deleted Reddit posts in which Platner appeared to mock military sexual assault victims, criticize police and make racially-charged comments about tipping.\nPlatner since apologized and blamed the posts on depression and PTSD after his military service in Afghanistan. He has vowed to stay in the race and has the backing of Sanders.\nJordan Wood, a Democratic rival in the primary and former chief of staff to Rep. Katie Porter, is calling on Platner to drop out.\n\"Graham Platner’s Reddit comments and Nazi SS Totenkopf tattoo are disqualifying and not who we are as Mainers or as Democrats,\" Wood said in a statement. \"With Donald Trump and his sycophants demonizing Americans, spewing hate and running roughshod over the Constitution, Democrats need to be able to condemn Trump’s actions with moral clarity. Graham Platner no longer can.\"\nPlatner said he believes the controversy is part of his life story, not disqualifying.\n\"I don’t look at this as a liability. I look at this as a life that I have lived, a journey that has been difficult, that has been full of struggle, that has also gotten me to where I am today,\" Platner told the AP. \"And I’m very proud of who I am.\"\nHe blamed \"establishment\" forces for amplifying the backlash to derail his campaign.\n\"Every second we spend talking about a tattoo I got in the Marine Corps is a second we don’t talk about Medicare for all,\" Platner said in the video.\nCLICK HERE TO GET THE FOX NEWS APP\nHe is running in a packed Democratic primary against Wood and two-term Gov. Janet Mills.\nGOP Sen. Susan Collins, who has held the seat for three decades, has not yet commented on the controversy.\nSanders and Collins did not immediately respond to Fox News Digital's request for comment.\nThe Associated Press contributed to this report.", "url": "https://www.foxnews.com/politics/maine-dem-senate-hopeful-backed-bernie-sanders-apologizes-nazi-style-tattoo-vows-stay-race"}

pred, probs = predict_article_DR(article)
print("Prediction:", pred)
print("Probabilities:", probs)

Prediction: Republican
Probabilities: {'Democrat': 0.492856987517007, 'Republican': 0.507143012482993}


In [14]:
article = {"title": "Trump reveals how US military is going to crack down on drug smugglers on land", 
           "source": "FOX", 
           "author": "Peter Pinedo", 
           "publication_date": "2025-10-22T00:00:00", 
           "content": "Speaking from the Oval Office on Wednesday evening, President Donald Trump said the U.S. will \"hit\" drug smugglers attempting to enter the U.S. by land after a series of lethal strikes on cartel boats at sea.\nThe president was questioned about his use of military force to crack down on drug smugglers in the Caribbean and Pacific following the eighth such strike in recent weeks.\nTrump acknowledged \"it is violent\" but said that \"every one of those boats that gets knocked out is saving 25,000 American lives.\"\n\"We have the greatest military in the world. We have the greatest weapons in the world. And you see a little bit of it there, one shot, every one dead center. And the only way you can’t feel bad about it is you realize … that every time you see that happen, you're saving 25,000 American lives.\nTRUMP’S WAR ON CARTELS ENTERS NEW PHASE AS EXPERTS PREDICT WHAT’S NEXT\n\"Whenever I see that, I say to myself, I just saved 25,000 lives.\"\nTrump said, after these strikes, \"there are very few boats traveling on the water, so now they'll come in by land to a lesser extent.\n\"And they will be hit on land also.\"\nPressed on whether he has legal authority to unleash strikes on drug smugglers on U.S. territory, Trump answered confidently, \"Yes, we do.\"\nTRUMP ADMIN ON PACE TO SHATTER DEPORTATION RECORD BY END OF FIRST YEAR: 'JUST THE BEGINNING'\n\"We have legal authority. We're allowed to do that,\" he said, noting that \"if we do it by land, we may go back to Congress.\n\"We'll probably go back to Congress and explain exactly what we're doing when we come to the land. We don't have to do that.\n\"This is a national security problem,\" Trump added. \"They killed 300,000 Americans last year, and that gives you legal authority.\n\"We will hit them very hard when they come in by land. They haven't experienced that yet, but now we're totally prepared to do that.\nTRUMP REFUSES TO RULE OUT STRIKING VENEZUELA. WHAT'S NEXT FOR TRUMP'S WAR ON DRUGS?\n\"Something very serious is going to happen. The equivalent of what's happening by sea. And we're going to Congress just to tell them what we're doing, just to keep them informed. But we have to do it for national security. We have to do it to save lives.\"\nCLICK HERE TO DOWNLOAD THE FOX NEWS APP\nSecretary of War Pete Hegseth announced Wednesday that, at Trump’s direction, the military carried out its first kinetic strike on \"narco-terrorists\" in the Eastern Pacific. This was the eighth such strike in recent weeks.", "url": "https://www.foxnews.com/politics/trump-reveals-how-us-military-going-to-crack-down-drug-smugglers-on-land"}


pred, probs = predict_article_DR(article)
print("Current Prediction:", pred)
print("Probabilities:", probs)

Current Prediction: Democrat
Probabilities: {'Democrat': 0.5232471749113038, 'Republican': 0.4767528250886962}


In [15]:
article = {"title": "Everything you need to know about Biden’s student loan forgiveness program", 
           "source": "CNN", 
           "author": "Katie Lobosco", 
           "publication_date": "2022-08-31T00:00:00", 
           "content": "The fate of President Joe Biden’s federal student loan forgiveness program, which promises to deliver up to $20,000 of debt relief for millions of borrowers, lies with the Supreme Court.\nThe justices heard arguments on February 28 in two cases concerning the forgiveness program, and a decision is expected by late June or early July.\nAbout 26 million people had already applied by the time a federal district court judge struck down the program on November 10, 2022 – prompting the government to stop taking applications. No debt has been canceled thus far.\nThe administration officially launched the application on October 17, 2022, following a brief “beta period” during which its team assessed whether tweaks were needed.\nIf the Supreme Court ultimately allows the program to move forward, not every student loan borrower is eligible for the debt relief. First, only federally held student loans qualify. Private student loans are excluded.\nSecond, high-income borrowers are generally excluded from receiving debt forgiveness. Individual borrowers who make less than $125,000 a year and married couples or heads of households who make less than $250,000 annually could see up to $10,000 of their federal student loan debt forgiven.\nIf a qualifying borrower also received a federal Pell grant while enrolled in college, the individual is eligible for up to $20,000 of debt forgiveness. Pell grants are awarded to millions of low-income students each year, based on factors including their family’s size and income and the cost charged by their college. These borrowers are also more likely to struggle to repay their student debt and end up in default.\nHere’s what else borrowers need to know about the proposed student loan forgiveness plan:\nWhat are the legal challenges to Biden’s forgiveness plan?\nThe Biden administration faced several lawsuits over the student loan forgiveness program, two of which have made it to the Supreme Court. The plaintiffs argue that the Department of Education is overstepping its authority.\nOne of the lawsuits was brought by six Republican-led states, headed by Nebraska, that argue that the student loan forgiveness program violates the separation of powers and the Administrative Procedure Act, a federal law that governs the process by which federal agencies issue regulations.\nA lower court judge dismissed this lawsuit on October 20, ruling that the plaintiffs did not have the legal standing to bring the challenge. In November, the 8th US Circuit Court of Appeals reversed that decision and blocked the program.\nThe other challenge that the Supreme Court heard was brought by two individual borrowers – Myra Brown and Alexander Taylor – who are not qualified for full debt relief forgiveness and who say they were denied an opportunity to comment on the secretary of education’s decision to provided targeted student loan debt relief to some.\nThe lawsuit was filed with the backing of a conservative group called the Job Creators Network Foundation. A federal judge in Texas ruled in favor of the plaintiffs, striking down the program on November 10.\nLawyers for the government say that Congress gave the secretary of education “expansive authority to alleviate the hardship that federal student loan recipients may suffer as a result of national emergencies,” like the Covid-19 pandemic, according to a memo from the Department of Justice.\nWhen will I receive my debt relief?\nIt’s unclear when, or if, borrowers will see debt relief under Biden’s program.\nAdministration officials expected to be able to grant relief before January, when payments were set resume after the pandemic-related pause expired. But now debt cancellation won’t occur until at least June when the Supreme Court is expected to issue a ruling.\nOn November 22, 2022, the Biden administration extended the pandemic-related pause on payments and interest a final time. Regardless of how the Supreme Court rules, student loan interest will resume on September 1 and payments will be due starting in October.\nA law passed in early June that addressees the debt ceiling prohibits another extension of the pause.\nThe White House has said that it has already approved 16 million applications for debt relief. The Department of Education will hold on to that information so it can quickly process those borrowers’ relief if the government prevails in court.\nIf and when the program moves forward, an estimated 8 million borrowers may receive debt relief automatically because the Department of Education already has their income on file.\nIf the government restarts taking applications, borrowers can apply online here: https://studentaid.gov/debt-relief/application.\nApplicants can expect to receive an email confirmation once their application is successfully submitted. Then, borrowers will be notified by their loan servicer when the debt cancellation has been applied to their account.\nBorrowers were expected to have until December 31, 2023, to submit an application.\nWhat kind of federal loans are eligible?\nThere are a variety of federal student loans and not all are eligible for relief if the program is allowed to proceed. Federal Direct Loans, including subsidized loans, unsubsidized loans, parent PLUS loans and graduate PLUS loans, are eligible.\nBut federal student loans that are guaranteed by the government but held by private lenders are not eligible unless the borrower applied to consolidate those loans into a Direct Loan by September 29, 2022.\nThe Department of Education initially said these privately held loans, many of which were made under the former Federal Family Education Loan program and Federal Perkins Loan program, would be eligible for the one-time forgiveness action – but reversed course last September when six Republican-led states sued the Biden administration, arguing that forgiving the privately held loans would financially hurt states and student loan servicers.\nDefaulted Federal Family Education Loans and defaulted Perkins Loans would be eligible for the debt relief even if they are privately held.\nWhat year is the income threshold based on?\nIf Biden’s program is allowed to move forward, eligibility is based on a borrower’s adjusted gross income for either tax year 2020 or 2021. Adjusted gross income can be lower than your total wages because it considers tax deductions and adjustments, like contributions made to a 401(k) retirement plan.\nA taxpayer’s adjusted gross income can be found on line 11 of IRS Form 1040.\nHow will the government know what my income was?\nThe Department of Education says it already has income information for nearly 8 million borrowers, likely because of financial aid forms or previously submitted income-driven repayment plan applications. If the program is allowed to move forward, those borrowers will automatically receive the debt relief if they meet the income requirement, unless they choose to opt out. The department has said it will email borrowers who will be considered for debt relief but don’t need to apply.\nMillions of other borrowers will need to apply for student loan forgiveness if the Department of Education doesn’t have their income information on file. When they submit the application, borrowers are required to self-attest that their income is under the eligibility threshold. They are required to certify that the information provided is accurate upon penalty of perjury.\nThe Biden administration has said that applicants who are “more likely to exceed the income cutoff” will be required to submit additional information, like a tax transcript. Officials expect that just 5% of borrowers with eligible federal student loans would not qualify due to the income threshold.\nWill I have to pay taxes on the amount of debt canceled?\nBorrowers will not have to pay federal income tax on the student loan debt forgiven, thanks to a provision in the American Rescue Plan Act that Congress passed in 2021.\nBut it’s possible that some borrowers may have to pay state income tax on the amount of debt forgiven. Most states will not tax the forgiveness as income.\nI’m a current student. Am I eligible for forgiveness?\nYes, some current students would be eligible under Biden’s plan if it’s allowed to take effect. Eligibility for borrowers who filed the Free Application for Federal Student Aid, known as the FAFSA, as an independent will be based on the individual’s own household income.\nEligibility for borrowers who are enrolled as dependent students, generally those under the age of 24, will be based on parental income for either 2020 or 2021.\nI have student debt from graduate school. Am I eligible for forgiveness?\nYes, if your income meets the eligibility threshold and the program is allowed to be implemented.\nI’m a parent and took out a Parent PLUS loan. Am I eligible?\nYes, you could be eligible if your income meets the threshold. A parent borrower with federal Parent PLUS loans for multiple children is still only eligible for up to $20,000 of loan forgiveness.\nBut a parent is only eligible for up to $20,000 in debt relief if he or she received a Pell grant for his or her own education. If only the child received a Pell grant, the parent is eligible for up to $10,000 in forgiveness.\nHow do I know if I ever received a Pell grant?\nMost borrowers can log in to Studentaid.gov to see if they received a Pell grant while enrolled in college. Information about Pell grants received is displayed on the account dashboard and on the My Aid page. This is also where borrowers can find out how much they owe and what kind of loans they have.\nBorrowers who received a Pell grant before 1994 won’t see their Pell grant information online, but they are still eligible for the $20,000 in student loan forgiveness.\nAs long as borrowers received at least one Pell grant, they are eligible.\nThe Biden administration has said that eligible borrowers who have received Pell grants will automatically receive the additional debt relief.\nAm I eligible for forgiveness if my loans are in default?\nYes, defaulted federal student loans would be eligible for debt relief under Biden’s program.\nFor borrowers who have a remaining balance on their defaulted student loans after the cancellation is applied, there will be an opportunity to get out of default once payments resume later this year as part of what the Department of Education is calling its “Fresh Start” initiative.\nHow will my payments change going forward?\nBorrowers who have debt remaining after either $10,000 or $20,000 is wiped away could see their monthly payment amounts recalculated if they are enrolled in a standard repayment plan. Under a standard repayment plan, borrowers pay a fixed amount that ensures loans are paid off within 10 years.\nBorrowers who are already enrolled in an income-driven repayment plan are not likely to see their monthly payment amounts change due to the forgiveness, because their payments are based on household income and family size.\nBorrowers have not been required to make payments on their federal student loans since March 2020 because of the government’s pandemic-related pause.\nWhat about Biden’s new income-driven repayment plan?\nAlong with Biden’s August announcement about canceling some federal student loan debt, he also said he would create a new plan that would make repayment more manageable for borrowers.\nThere are currently several repayment plans available for federal student loan borrowers that lower monthly payments by capping them at a portion of their income.\nThe new income-driven repayment plan proposal will cap payments at 5% of a borrower’s discretionary income, down from 10% that is offered in most current plans, as well as reduce the amount of income that is considered discretionary. It would also forgive remaining balances after 10 years of repayment, instead of 20 years.\nBiden is also proposing that the new plan cover the borrower’s unpaid monthly interest. This could be very helpful for people whose monthly payments are so low that they don’t cover their monthly interest charge and end up seeing their balances explode, growing larger than what was originally borrowed.\nThe new plan is currently going through a formal rulemaking process, and the Department of Education has said it expects provisions of the new plan to take effect later this year.\nCan I get a refund for what I paid during the pandemic pause?\nYes. Borrowers have not been required to make payments on their federal student loans since March 13, 2020, because of the pandemic-related pause. But if borrowers did make payments, they are allowed to contact their loan servicer to request a refund.\nThis story has been updated with additional information.", "url": "https://www.cnn.com/2022/08/31/politics/biden-student-loan-forgiveness-faq/index.html"}

pred, probs = predict_article_DR(article)
print("Current Prediction:", pred)
print("Probabilities:", probs)


Current Prediction: Republican
Probabilities: {'Democrat': 0.47794247895511144, 'Republican': 0.5220575210448886}


In [16]:
from joblib import dump, load
import json, numpy as np, re, os

os.makedirs("models", exist_ok=True)
dump(pipe, "models/political_affiliation.joblib")
with open("models/party_labels.json", "w") as f:
    json.dump({"classes": labels}, f)
print("Saved: models/political_affiliation.joblib")

Saved: models/political_affiliation.joblib
